## Plze?sk? kraj ?koly

In [ ]:
import pandas
import xml.etree.ElementTree as ET
import re
xml_soubor = 'plzensky_puvodni.xml'

In [ ]:

tree = ET.parse('plzensky_puvodni.xml')
root = tree.getroot()

skoly = []
for entry in root.findall('PravniSubjekt'):
    skola_group = entry.find('SkolyZarizeni')
    for skola in skola_group.findall('SkolaZarizeni'):
        izo = skola.find('IZO').text
        typ = skola.find('SkolaDruhTyp').text
        nazev = skola.find('SkolaPlnyNazev').text
        adresa = skola.find('SkolaMistaVykonuCinnosti')
        actual_ad = adresa.find('SkolaMistoVykonuCinnosti')
        adresa1 =  actual_ad.find('MistoAdresa1').text
        adresa2 =  actual_ad.find('MistoAdresa2').text
        adresa3 =  actual_ad.find('MistoAdresa3').text
        if adresa3 == None:
            adresa3 = adresa2
        #izoluji n?zev obce regul?rn?m v?razem a ukl?d?m do sloupce obec
        pattern1 = r' \d\d '
        list_obec = re.split(pattern1, adresa3)
        obec = list_obec[-1]
        #izoluji ps?
        pattern2 = r' [A-Z????????]'
        list_obec = re.split(pattern2, adresa3)
        PSC = list_obec[0]
        #p?id?v?m sloupec kraj
        kraj = 'Plze?sk? kraj'
        
        if typ == 'C00':  # ??seln?k pro S?
            obor = skola.find('SkolaOboryVzdelani')
            actual_obor = obor.find('SkolaOborVzdelani')
            obor_kod = actual_obor.find('OborKod').text
            obor_nazev = actual_obor.find('OborNazev').text
            obor_forma = actual_obor.find('FormaVzdelavani').text
            obor_delka = actual_obor.find('DelkaVzdelavani').text
            obor_kapacita = actual_obor.find('OborKapacita').text
            
            skoly.append([izo, nazev, typ, adresa1, adresa2, adresa3, PSC, obec, kraj, obor_kod, obor_nazev, obor_forma, obor_delka, obor_kapacita])

sloupce = ['id', 'nazev', 'typ', 'adresa1', 'adresa2', 'adresa3', 'PSC', 'obec', 'kraj', 'obor_kod', 'obor_nazev', 'obor_forma', 'obor_delka', 'obor_kapacita']
df_skoly_plzensky = pandas.DataFrame(skoly, columns = sloupce)

pandas.set_option('display.max_rows', None)  # chci vid?t v?echny ??dky
df_skoly_plzensky

In [ ]:
df_skoly_plzensky['adresa3'].isna().sum()


In [ ]:
df_skoly_plzensky['adresa3'].unique()

p?id?v?m sloupec okres a PSC

In [ ]:
vazba_obec_psc_okres = pandas.read_csv("vazba_obec_psc_okres.csv")



In [ ]:
df_plzensky_kraj_final = pandas.merge(df_skoly_plzensky, vazba_obec_psc_okres, on=['PSC', 'obec'], how='left')


In [ ]:
df_plzensky_kraj_final['okres'].isna().sum()

In [ ]:
pandas.set_option('display.max_rows', None)
df_plzensky_kraj_final

In [ ]:
df_plzensky_kraj_final.loc[30, 'okres'] = 'Plze?-jih'

In [ ]:
df_plzensky_kraj_final['okres'].isna().sum()

In [ ]:
#ukl?d?m do csv
df_plzensky_kraj_final.to_csv('skoly_Plzensky_final.csv', index = None)